##### 05 - Incremental Processing (MERGE / UPSERT / CDC)
This notebook demonstrates production patterns for incremental data loading:
1. **MERGE (Upsert)** - Update existing + insert new records
2. **SCD Type 1** - Overwrite with latest values
3. **SCD Type 2** - Track history with effective dates (simplified)
4. **Append-only** - Insert new facts without updating old ones
5. **Delta Time Travel** - Query previous versions for auditability

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime, timedelta
import random

SILVER_SCHEMA = "ecommerce_silver"
GOLD_SCHEMA = "ecommerce_gold"
INCREMENTAL_SCHEMA = "ecommerce_incremental"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {INCREMENTAL_SCHEMA}")

DataFrame[]

##### 1. Simulate Incoming CDC Data
Simulate a batch of changed records that arrived since the last run:
- Updated customer addresses (SCD Type 1)
- New orders (append)
- Price changes for products (MERGE)

In [0]:
df_existing_customers = spark.table(f"{SILVER_SCHEMA}.customers")
sample_customer_ids = [row.customer_id for row in df_existing_customers.limit(500).collect()]

CITIES_NEW = [("Gurgaon", "Haryana"), ("Noida", "Uttar Pradesh"),
              ("Chandigarh", "Chandigarh"), ("Mysore", "Karnataka")]

random.seed(99)

address_changes = []
for cid in random.sample(sample_customer_ids, 200):
    city, state = random.choice(CITIES_NEW)
    address_changes.append((cid, city, state, "active", str(datetime.now())))

df_customer_updates = spark.createDataFrame(
    address_changes,
    ["customer_id", "city", "state", "status", "_update_timestamp"]
)

print(f"Simulated {df_customer_updates.count()} customer address changes")
df_customer_updates.show(5, truncate=False)


Simulated 200 customer address changes
+-----------+----------+-------------+------+--------------------------+
|customer_id|city      |state        |status|_update_timestamp         |
+-----------+----------+-------------+------+--------------------------+
|CUST-000207|Chandigarh|Chandigarh   |active|2026-05-01 13:17:09.197873|
|CUST-000195|Noida     |Uttar Pradesh|active|2026-05-01 13:17:09.197894|
|CUST-000103|Noida     |Uttar Pradesh|active|2026-05-01 13:17:09.197905|
|CUST-000307|Chandigarh|Chandigarh   |active|2026-05-01 13:17:09.197914|
|CUST-000092|Mysore    |Karnataka    |active|2026-05-01 13:17:09.197922|
+-----------+----------+-------------+------+--------------------------+
only showing top 5 rows


##### 2. MERGE Pattern - Customer Address Updates (SCD Type 1)
This overwrites the existing record with the latest values.
In production, this handles late-arriving data and corrections.

In [0]:
target_table = f"{INCREMENTAL_SCHEMA}.customers_scd1"

(
    spark.table(f"{SILVER_SCHEMA}.customers")
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

dt_customers = DeltaTable.forName(spark, target_table)

print(f"Before MERGE - sample record:")
dt_customers.toDF().filter(F.col("customer_id") == sample_customer_ids[0]).show(truncate=False)

(
    dt_customers.alias("target")
    .merge(
        df_customer_updates.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdate(set={
        "city": "source.city",
        "state": "source.state",
        "status": "source.status",
        "_silver_timestamp": F.current_timestamp(),
    })
    .execute()
)

print(f"\nAfter MERGE - same record:")
dt_customers.toDF().filter(F.col("customer_id") == sample_customer_ids[0]).show(truncate=False)

merge_metrics = dt_customers.history(1).select("operationMetrics").collect()[0][0]
print(f"MERGE Metrics:")
print(f"  Rows updated:  {merge_metrics.get('numTargetRowsUpdated', 'N/A')}")
print(f"  Rows inserted: {merge_metrics.get('numTargetRowsInserted', 'N/A')}")
print(f"  Rows deleted:  {merge_metrics.get('numTargetRowsDeleted', 'N/A')}")


Before MERGE - sample record:
+-----------+----------+---------+--------------------------+--------------+---+------+-----------+-----------+--------+-----------+--------------------------+
|customer_id|first_name|last_name|email                     |phone         |age|city  |state      |signup_date|status  |full_name  |_silver_timestamp         |
+-----------+----------+---------+--------------------------+--------------+---+------+-----------+-----------+--------+-----------+--------------------------+
|CUST-000001|Rohan     |Singh    |rohan.singh760@outlook.com|+91-7802439256|26 |Mumbai|Maharashtra|2024-02-22 |inactive|Rohan Singh|2026-05-01 06:47:05.384918|
+-----------+----------+---------+--------------------------+--------------+---+------+-----------+-----------+--------+-----------+--------------------------+


After MERGE - same record:
+-----------+----------+---------+--------------------------+--------------+---+------+-----------+-----------+--------+-----------+---------

##### 3. SCD Type 2 - Customer History Tracking
Maintains full history by closing old records (setting `effective_end`)
and inserting new versions with `is_current = true`.

In [0]:
from pyspark.sql.types import BooleanType

scd2_table = f"{INCREMENTAL_SCHEMA}.customers_scd2"

df_initial = (
    spark.table(f"{SILVER_SCHEMA}.customers")
    .withColumn("effective_start", F.current_date())
    .withColumn("effective_end", F.lit("9999-12-31").cast("date"))
    .withColumn("is_current", F.lit(True).cast(BooleanType()))
    .withColumn("version", F.lit(1))
)

(
    df_initial.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(scd2_table)
)

# COMMAND ----------

dt_scd2 = DeltaTable.forName(spark, scd2_table)

updates_for_scd2 = df_customer_updates.limit(50)

current_snapshot = dt_scd2.toDF().filter("is_current = true")

staged_temp_table = f"{INCREMENTAL_SCHEMA}._scd2_staged_tmp"

(
    updates_for_scd2.alias("u")
    .join(current_snapshot.alias("t"), "customer_id")
    .filter(
        (F.col("u.city") != F.col("t.city"))
        | (F.col("u.state") != F.col("t.state"))
    )
    .select(
        F.col("u.customer_id").alias("customer_id"),
        F.col("t.first_name").alias("first_name"),
        F.col("t.last_name").alias("last_name"),
        F.col("t.email").alias("email"),
        F.col("t.phone").alias("phone"),
        F.col("t.age").alias("age"),
        F.col("u.city").alias("city"),
        F.col("u.state").alias("state"),
        F.col("t.signup_date").alias("signup_date"),
        F.col("u.status").alias("status"),
        F.col("t.full_name").alias("full_name"),
    )
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(staged_temp_table)
)

staged = spark.table(staged_temp_table)

print(f"Staged updates with actual changes: {staged.count()}")

new_records = (
    staged
    .withColumn("_silver_timestamp", F.current_timestamp())
    .withColumn("effective_start", F.current_date())
    .withColumn("effective_end", F.lit("9999-12-31").cast("date"))
    .withColumn("is_current", F.lit(True))
    .withColumn("version", F.lit(2))
)

(
    dt_scd2.alias("target")
    .merge(
        staged.select("customer_id").alias("source"),
        "target.customer_id = source.customer_id AND target.is_current = true"
    )
    .whenMatchedUpdate(set={
        "effective_end": F.current_date(),
        "is_current": F.lit(False),
    })
    .execute()
)

new_records.write.format("delta").mode("append").saveAsTable(scd2_table)

spark.sql(f"DROP TABLE IF EXISTS {staged_temp_table}")

scd2_total = spark.table(scd2_table)
print(f"SCD2 total records: {scd2_total.count()}")
print(f"Current records:    {scd2_total.filter('is_current = true').count()}")
print(f"Historical records: {scd2_total.filter('is_current = false').count()}")

print("\nExample - customer with version history:")
sample = scd2_total.filter(F.col("version") > 1).select("customer_id").first()
if sample:
    scd2_total.filter(F.col("customer_id") == sample.customer_id).orderBy("version").show(truncate=False)


Staged updates with actual changes: 50
SCD2 total records: 10050
Current records:    10000
Historical records: 50

Example - customer with version history:
+-----------+----------+---------+--------------------------+--------------+---+---------+---------+-----------+------+-----------+--------------------------+---------------+-------------+----------+-------+
|customer_id|first_name|last_name|email                     |phone         |age|city     |state    |signup_date|status|full_name  |_silver_timestamp         |effective_start|effective_end|is_current|version|
+-----------+----------+---------+--------------------------+--------------+---+---------+---------+-----------+------+-----------+--------------------------+---------------+-------------+----------+-------+
|CUST-000043|Rohan     |Mehta    |rohan.mehta453@hotmail.com|+91-9213838010|50 |Ahmedabad|Gujarat  |2024-08-30 |active|Rohan Mehta|2026-05-01 06:47:05.384918|2026-05-01     |2026-05-01   |false     |1      |
|CUST-000043

#### 4. Append-Only Pattern - New Orders
Fact tables typically use append-only ingestion with a high watermark
to avoid re-processing already loaded records.

In [0]:
df_orders = spark.table(f"{SILVER_SCHEMA}.orders")

last_watermark = "2025-06-01"
new_watermark = "2025-07-01"

df_incremental_orders = (
    df_orders
    .filter(
        (F.col("order_date") > F.lit(last_watermark))
        & (F.col("order_date") <= F.lit(new_watermark))
    )
)

append_target = f"{INCREMENTAL_SCHEMA}.orders_incremental"

(
    df_orders.filter(F.col("order_date") <= last_watermark)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(append_target)
)

before_count = spark.table(append_target).count()

(
    df_incremental_orders.write
    .format("delta")
    .mode("append")
    .saveAsTable(append_target)
)

after_count = spark.table(append_target).count()

print(f"Watermark range: {last_watermark} -> {new_watermark}")
print(f"Records before:  {before_count:,}")
print(f"New records:     {df_incremental_orders.count():,}")
print(f"Records after:   {after_count:,}")


Watermark range: 2025-06-01 -> 2025-07-01
Records before:  20,907
New records:     4,049
Records after:   24,956


##### 5. Delta Time Travel - Audit Previous States
One of Delta Lake's most powerful features: querying data at any previous version.

In [0]:
dt_target = DeltaTable.forName(spark, f"{INCREMENTAL_SCHEMA}.customers_scd1")

print("Version History:")
dt_target.history().select("version", "timestamp", "operation",
                           "operationMetrics.numOutputRows").show(truncate=False)

# COMMAND ----------

v0_count = spark.sql(
    f"SELECT COUNT(*) AS c FROM {INCREMENTAL_SCHEMA}.customers_scd1 VERSION AS OF 0"
).collect()[0]["c"]

v1_count = spark.sql(
    f"SELECT COUNT(*) AS c FROM {INCREMENTAL_SCHEMA}.customers_scd1 VERSION AS OF 1"
).collect()[0]["c"]

current_count = spark.table(f"{INCREMENTAL_SCHEMA}.customers_scd1").count()

print(f"Version 0 (initial load): {v0_count:,} rows")
print(f"Version 1 (after MERGE):  {v1_count:,} rows")
print(f"Current version:          {current_count:,} rows")

Version History:
+-------+-------------------+---------------------------------+-------------+
|version|timestamp          |operation                        |numOutputRows|
+-------+-------------------+---------------------------------+-------------+
|6      |2026-05-01 13:17:37|MERGE                            |200          |
|5      |2026-05-01 13:17:32|CREATE OR REPLACE TABLE AS SELECT|10000        |
|4      |2026-05-01 13:02:33|MERGE                            |200          |
|3      |2026-05-01 13:02:27|CREATE OR REPLACE TABLE AS SELECT|10000        |
|2      |2026-05-01 13:01:58|MERGE                            |200          |
|1      |2026-05-01 13:01:50|CREATE OR REPLACE TABLE AS SELECT|10000        |
|0      |2026-05-01 12:58:56|CREATE OR REPLACE TABLE AS SELECT|10000        |
+-------+-------------------+---------------------------------+-------------+

Version 0 (initial load): 10,000 rows
Version 1 (after MERGE):  10,000 rows
Current version:          10,000 rows


##### 6. Idempotent Reprocessing with MERGE
Running the same MERGE twice produces the same result - critical for production pipelines.

In [0]:
before_rerun = dt_customers.toDF().count()

(
    dt_customers.alias("target")
    .merge(
        df_customer_updates.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdate(set={
        "city": "source.city",
        "state": "source.state",
        "status": "source.status",
        "_silver_timestamp": F.current_timestamp(),
    })
    .execute()
)

after_rerun = dt_customers.toDF().count()

print(f"Before re-run: {before_rerun:,} rows")
print(f"After re-run:  {after_rerun:,} rows")
print(f"Idempotent: {before_rerun == after_rerun}")


Before re-run: 10,000 rows
After re-run:  10,000 rows
Idempotent: True


##### 7. Vacuum & Optimize
Maintenance operations to keep Delta tables performant.

In [0]:
spark.sql(f"OPTIMIZE {INCREMENTAL_SCHEMA}.customers_scd1")
print("OPTIMIZE complete - small files compacted.")

spark.sql(f"VACUUM {INCREMENTAL_SCHEMA}.customers_scd1 RETAIN 168 HOURS")
print("VACUUM complete - old files cleaned up (retaining 7 days).")

print("Incremental processing complete! Proceed to notebook 06_analytics_queries.")


OPTIMIZE complete - small files compacted.
VACUUM complete - old files cleaned up (retaining 7 days).
Incremental processing complete! Proceed to notebook 06_analytics_queries.
